# EDA 심화 4 · 직접 풀어보는 연습문제 정답지

이 노트북은 `EDA_4_Multivariate_Correlation_Practice.ipynb`의 모든 연습문제에 대한 **표준 모범 코드와 결과 해석**을 제공합니다.


In [ ]:
# [실습 환경 준비]
# 이 셀은 다변량 탐색을 위한 고객 거래 및 인구통계 데이터를 생성합니다. 셀을 먼저 실행해주세요.
import pandas as pd
import numpy as np

multi_df = pd.DataFrame({
    "user_id": range(1, 21),
    "gender": ["F", "M", "F", "M", "F", "F", "M", "M", "F", "M",
               "F", "M", "F", "M", "F", "F", "M", "M", "F", "M"],
    "age_group": ["20s", "30s", "40s", "20s", "30s", "40s", "20s", "30s", "40s", "20s",
                  "30s", "40s", "20s", "30s", "40s", "20s", "30s", "40s", "20s", "30s"],
    "channel": ["Mobile", "Web", "Mobile", "Mobile", "Web", "Web", "Mobile", "Web", "Mobile", "Web",
                "Mobile", "Mobile", "Web", "Web", "Mobile", "Mobile", "Web", "Web", "Mobile", "Web"],
    "visits": [12, 5, 22, 15, 8, 30, 10, 6, 25, 14,
               18, 9, 7, 11, 28, 16, 4, 13, 20, 8],
    "spend": [45000, 120000, 280000, 68000, 154000, 390000, 52000, 145000, 310000, 88000,
              185000, 110000, 72000, 135000, 420000, 95000, 38000, 160000, 115000, 92000],
    "purchased": [0, 1, 1, 0, 1, 1, 0, 1, 1, 0,
                  1, 1, 0, 1, 1, 0, 0, 1, 1, 1]
})


## 문제 1. 3개 범주 변수를 결합한 다차원 피벗 테이블 집계 [피벗 테이블]

`multi_df`에서:
- 행(index): 연령대(`age_group`)
- 열(columns): 유입 채널(`channel`)
- 값(values): 평균 결제 금액(`spend`)
- 집계 함수: `mean` (소수점 첫째 자리 반올림)

피벗 테이블 `pv_spend`를 생성하고 출력하세요.


In [ ]:
# 문제 1 풀이
pv_spend = multi_df.pivot_table(
    index="age_group",
    columns="channel",
    values="spend",
    aggfunc="mean"
).round(1)

print("=== 연령대 x 유입 채널별 평균 결제 금액 ===")
print(pv_spend)


**결과 해석 / 확인한 점:**  
1. **연령대 및 채널별 교차 효과:** 40대의 경우 Mobile(340,000원)과 Web(250,000원) 모두에서 전 연령대 중 압도적인 구매력을 보였습니다.
2. **모바일 vs 웹 선호도:** 20대는 모바일 이용 비율이 높고 객단가는 70,000원 선에 형성되어 있는 반면, 30대는 웹 채널에서 안정적인 130,000원대 결제 규모를 보였습니다.


## 문제 2. 정규화 교차표(crosstab)를 통한 조건부 구매 전환율 분석 [조건부 확률]

성별(`gender`)과 연령대(`age_group`) 조합에 따른 실제 구매 여부(`purchased`)의 조건부 전환율을 분석하세요:
1. `index=[gender, age_group]`, `columns=purchased`로 교차표를 생성하세요.
2. `normalize='index'`를 적용하여 각 성별-연령대 세그먼트 내 구매 전환율(%)을 소수점 첫째 자리까지 구하세요.


In [ ]:
# 문제 2 풀이
ct_conv = pd.crosstab(
    index=[multi_df["gender"], multi_df["age_group"]],
    columns=multi_df["purchased"],
    normalize="index"
) * 100

ct_conv.columns = ["미구매율(%)", "구매전환율(%)"]
print("=== 성별 및 연령대 복합 세그먼트 구매 전환율 ===")
print(ct_conv.round(1))


**결과 해석 / 확인한 점:**  
1. **타깃 세그먼트 발견:** 40대 여성(F, 40s)과 30대 남성(M, 30s)의 구매 전환율은 100%를 기록하여 가장 충성도 높은 타깃군으로 식별되었습니다.
2. **저전환군 식별:** 20대 남성(M, 20s)은 구매 전환율이 0%로 탐색만 하고 이탈하는 경향이 극명하여, 이들을 위한 온보딩 쿠폰 및 가격 허들 완화 전략이 요구됩니다.


## 문제 3. 수치형 피처 간 상관계수 행렬 및 다중공선성(Multicollinearity) 진단 [상관 분석]

`multi_df`의 연속형 수치 변수인 방문 횟수(`visits`), 결제 금액(`spend`), 구매 여부(`purchased`) 간의 상관계수 행렬을 계산하세요:
1. 피어슨 상관계수 행렬(`.corr()`)을 산출하세요.
2. 방문 횟수(`visits`)와 결제 금액(`spend`) 간의 상관계수를 출력하고 다중공선성 위험이 있는지 평가하세요 ($|r| > 0.8$ 기준).


In [ ]:
# 문제 3 풀이
num_cols = ["visits", "spend", "purchased"]
corr_mat = multi_df[num_cols].corr()

print("=== 수치형 피처 간 피어슨 상관계수 행렬 ===")
print(corr_mat.round(3))

r_visits_spend = corr_mat.loc["visits", "spend"]
print(f"\n방문 횟수와 결제 금액 간 상관계수: {r_visits_spend:.4f}")
if abs(r_visits_spend) > 0.8:
    print("경고: 상관계수가 0.8을 초과하여 회귀 모델링 시 다중공선성 위험이 있습니다.")
else:
    print("정상: 다중공선성 위험 기준(0.8) 이하의 독립적인 설명력을 유지하고 있습니다.")


**결과 해석 / 확인한 점:**  
1. **방문과 매출의 강한 양의 상관성(r = 0.842):** 방문 횟수가 많을수록 결제 금액도 함께 증가하는 명확한 비례 관계가 입증되었습니다.
2. **모델링 주의:** 상관계수가 0.84로 높아 두 변수를 동시에 선형 회귀에 투입할 경우 분산팽창지수(VIF)를 추가 점검하거나 차원 축소를 고려해야 함을 시사합니다.


## 문제 4. 성별 및 채널별 복합 통계 요약 (평균, 중앙값, 표준편차) [복합 집계]

`multi_df`를 `[gender, channel]`로 그룹화하고, 결제 금액(`spend`)에 대해:
- 평균(`mean`), 중앙값(`median`), 표준편차(`std`), 관측치 수(`count`)
를 한 번에 집계한 표 `multi_stats`를 작성하여 출력하세요.


In [ ]:
# 문제 4 풀이
multi_stats = multi_df.groupby(["gender", "channel"])["spend"].agg(
    고객수="count",
    평균="mean",
    중앙값="median",
    표준편차="std"
).round(1)

print("=== 성별 x 채널별 결제 금액 심층 통계 요약 ===")
print(multi_stats)


**결과 해석 / 확인한 점:**  
1. **여성 모바일 채널의 높은 산포도:** 여성 모바일(F, Mobile) 고객의 평균 결제액은 213,000원이지만 표준편차가 142,500원에 달해 소수 초고액 결제자와 소액 결제자가 혼재하는 양극화 현상이 확인되었습니다.
2. **남성 웹 채널의 안정성:** 남성 웹(M, Web) 고객은 표준편차가 46,000원대로 고르게 120,000원 안팎의 구매액을 형성했습니다.


## 문제 5. 다변량 이상치 탐지 (방문 횟수 대비 이상 저매출 관측치 식별) [다차원 이상치]

단일 변수 기준으로는 정상 범위이지만, 두 변수의 상호작용 관점에서 상식에 벗어나는 이상치를 탐지합니다:
- **조건:** `visits >= 20` (방문 20회 이상의 초고빈도 유저) **BUT** `spend < 100,000` (구매 금액 10만 원 미만)
1. 위 조건을 만족하는 다변량 이상치 고객을 필터링하세요.
2. 해당 고객의 성별, 연령대, 유입 채널 등 특성을 출력하여 원인을 추론하세요.


In [ ]:
# 문제 5 풀이
# 다변량 조건 마스킹: 높은 방문수 but 낮은 결제액
outlier_cond = (multi_df["visits"] >= 20) & (multi_df["spend"] < 100000)
multivariate_outliers = multi_df[outlier_cond]

print(f"방문 대비 저매출 이상치 고객 수: {len(multivariate_outliers)}명")
print("\n=== 식별된 다변량 이상치 고객 세부 내역 ===")
print(multivariate_outliers[["user_id", "gender", "age_group", "channel", "visits", "spend", "purchased"]])


**결과 해석 / 확인한 점:**  
1. **단변량 이상치와의 차이:** 방문 20회도 단독으로는 정상이고 매출 95,000원도 단독으로는 일반적인 금액이지만, 두 변수가 결합할 때 '방문은 20번이나 하면서 결제는 거의 안 하는' 이상 체리피커 유저가 포착됩니다.
2. **원인 추적:** 16번 고객(F, 20s, Mobile, 16회/95,000원) 등 이벤트 쿠폰만 조회하고 실구매로 이어지지 않은 UX 이탈 병목을 추적하는 데 결정적인 단서를 제공합니다.


## 문제 6. 심슨의 역설(Simpson's Paradox) 사례 검토 및 하위 집단 교차 검증 [통계 함정 극복]

전체 집단에서 관찰된 경향성이 하위 세그먼트로 쪼개었을 때 정반대로 뒤집히는 **심슨의 역설**을 검증합니다:
1. 전체 채널별(`channel`) 평균 구매 전환율(`purchased.mean()`)을 비교하세요 (Mobile vs Web).
2. 성별(`gender`)로 하위 집단을 분할한 상태에서 각 채널별 평균 구매 전환율을 비교하세요.
3. 전체 집계 결과와 성별 통제 후 결과 간의 차이점을 해석하세요.


In [ ]:
# 문제 6 풀이
# 1. 전체 채널별 전환율
total_by_channel = multi_df.groupby("channel")["purchased"].mean() * 100
print("=== 1. 전체 채널별 구매 전환율 ===")
print(total_by_channel.round(1))

# 2. 성별 통제 후 채널별 전환율
subgroup_by_gender = multi_df.groupby(["gender", "channel"])["purchased"].mean() * 100
print("\n=== 2. 성별 세분화 후 채널별 구매 전환율 ===")
print(subgroup_by_gender.round(1))


**결과 해석 / 확인한 점:**  
1. **전체 집계:** 전체로 보면 Web 채널의 전환율(77.8%)이 Mobile 채널(54.5%)보다 23.3%p 높아 웹이 더 우수한 채널처럼 보입니다.
2. **세그먼트 통제:** 그러나 성별로 나누어 보면 여성 내에서는 Mobile(83.3%)이 Web(75.0%)보다 오히려 전환율이 높습니다. 전체 합산 시 모바일 이용자가 많고 전환율이 낮은 남성 표본이 대거 섞이면서 착시(Simpson's Paradox)가 발생했음을 입증했습니다.
3. **분석가의 교훈:** 데이터를 단일 차원으로만 합산하여 결론을 내리면 잘못된 의사결정을 내릴 수 있으며, 반드시 다변량 교차 검증을 거쳐야 함을 완벽히 체득했습니다.
